# Add an IIIF facsimile and measure zones to one MEI file

**Workflow 1 — handle MEI files.** This notebook starts with one clean MEI file, connects it to a source-page image, detects measure regions on that picture, and writes the regions plus facsimile references into a new MEI file.

The example is `test_corpus/Buxtehude-Anhang-S._185_musicxml_verovio.mei` in this repository: 17 measures and no facsimile data yet. **Clean** here means no `<facsimile>`, `<surface>`, `<zone>`, or measure `@facs` yet; schema and editorial consistency are later checks. Its Bayerische Staatsbibliothek page is <https://digitale-sammlungen.de/en/view/bsb00023199?page=185>.

Network calls and writes stay off until you set `RUN_IIIF_INTEGRATION = True`. Generated files go to `converted_mei/iiif_tutorial/` (gitignored). The original in `test_corpus/` is not overwritten.

Continue afterwards with [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb).

In [1]:
# You can leave this cell unchanged.
try:
    from camat import (
        DETECTOR_URL,
        IIIF_IMAGE_URL_TEMPLATE,
        detect_and_integrate_mei,
        display_path,
        download_facsimile_image,
        format_facsimile_summary,
        parse_bsb_viewer_url,
        parse_graphic_from_output_mei,
        read_facsimile_model,
        read_image_size,
        resolve_repo_path,
        sha256_bytes,
        sha256_file,
        stage_mei_copy,
    )
except ModuleNotFoundError:
    import setup_camat
    from camat import (
        DETECTOR_URL,
        IIIF_IMAGE_URL_TEMPLATE,
        detect_and_integrate_mei,
        display_path,
        download_facsimile_image,
        format_facsimile_summary,
        parse_bsb_viewer_url,
        parse_graphic_from_output_mei,
        read_facsimile_model,
        read_image_size,
        resolve_repo_path,
        sha256_bytes,
        sha256_file,
        stage_mei_copy,
    )

import requests

## 1. Configure the job

Set `SOURCE_MEI` to a clean MEI file. Identify the facsimile page in either or both of these ways:

- **`ARCHIVE_URL`** — a Digitale Sammlungen viewer link such as `.../view/bsb00023199?page=185`. CAMAT reads the BSB id and page number and names the working copy `bsb00023199_00185.mei`.
- **`IIIF_IMAGE_URL`** — a full IIIF image URL. When this is set, CAMAT downloads that address instead of building one from the working filename. Use this for non-BSB images, or when the filename should not determine the IIIF link.

If both are set, `ARCHIVE_URL` still controls the working filename and `IIIF_IMAGE_URL` controls the image that is fetched and written as `<graphic @target>`.

Review the plan in the next cell, then set `RUN_IIIF_INTEGRATION = True`.

In [4]:
# Clean MEI in this repository (17 measures, no facsimile yet).
SOURCE_MEI = "test_corpus/Buxtehude-Anhang-S._185_musicxml_verovio.mei"

# Bayerische Staatsbibliothek viewer page for this source.
# Used to name the working copy unless you leave this empty and set IIIF_IMAGE_URL.
ARCHIVE_URL = "https://digitale-sammlungen.de/en/view/bsb00023199?page=185"

# Optional full IIIF image URL. When set, this is downloaded instead of a
# URL built from the MEI filename. Example:
# "https://api.digitale-sammlungen.de/iiif/image/v2/bsb00023199_00185/full/4134,/0/default.jpg"
IIIF_IMAGE_URL = ""

# Generated files (staged MEI, image, annotations, output). Gitignored.
TARGET_DIR = "converted_mei/iiif_tutorial"

TARGET_DPI = 500
PAGE_WIDTH_MM = 210.0
MINIMUM_MEASURES = 5
MAX_MEASURE_MISMATCH = 1
TIMEOUT = 180

# Network calls, detector upload, and all writes stay off until this is True.
RUN_IIIF_INTEGRATION = False

OVERWRITE_STAGED_MEI = False
OVERWRITE_OUTPUT = False
REUSE_ANNOTATIONS = False

## 2. Review the plan

This cell only reads the source and prints the paths that would be written. It does not download or change files.

The **measure detector** (`DETECTOR_URL`, currently `https://measure-detector.edirom.de/upload`) is Edirom's DOMD service: a neural-network model that finds measure boxes on the facsimile *picture*. It does not read the MEI. CAMAT uploads the page image, receives bounding boxes, and writes them as `<zone type="measure">` elements, then points each encoded `<measure>` at its zone with `@facs`.

In [5]:
source_mei = resolve_repo_path(SOURCE_MEI)
target_dir = resolve_repo_path(TARGET_DIR)
archive_url = ARCHIVE_URL.strip()
iiif_image_url = IIIF_IMAGE_URL.strip() or None

if not archive_url and not iiif_image_url:
    raise ValueError("Set ARCHIVE_URL and/or IIIF_IMAGE_URL so CAMAT can find the facsimile page.")

if archive_url:
    bsb_id, page_number, target_stem = parse_bsb_viewer_url(archive_url)
else:
    bsb_id = page_number = None
    target_stem = source_mei.stem

staged_mei = target_dir / f"{target_stem}.mei"
image_dir = target_dir / "img"
image_path = image_dir / f"{target_stem}.jpg"
annotation_path = target_dir / f"{target_stem}_measure_annotations.xml"
final_mei = target_dir / f"{target_stem}_facs_zones.mei"

if not source_mei.is_file():
    raise FileNotFoundError(source_mei)

source_model = read_facsimile_model(source_mei, allow_missing_facsimile=True)
if source_model["has_facsimile"]:
    raise ValueError("SOURCE_MEI already contains facsimile integration.\n" + format_facsimile_summary(source_model))

print(format_facsimile_summary(source_model))
print(f"BSB id:       {bsb_id or '(not derived; using IIIF_IMAGE_URL)'}")
print(f"Page number:  {page_number if page_number is not None else '(n/a)'}")
print(f"Working stem: {target_stem}")
print(f"IIIF image:   {iiif_image_url or '(built from working stem after download)'}")
print(f"Staged MEI:   {display_path(staged_mei)}")
print(f"Image:        {display_path(image_path)}")
print(f"Annotation:   {display_path(annotation_path)}")
print(f"Final output: {display_path(final_mei)}")
print(f"Detector:     {DETECTOR_URL}")
print(f"Run enabled:  {RUN_IIIF_INTEGRATION}")

MEI:             test_corpus/Buxtehude-Anhang-S._185_musicxml_verovio.mei
Viewer mode:     score only
Measures:        17
Facsimile:       unavailable (No <facsimile> found in test_corpus/Buxtehude-Anhang-S._185_musicxml_verovio.mei)
BSB id:       bsb00023199
Page number:  185
Working stem: bsb00023199_00185
IIIF image:   (built from working stem after download)
Staged MEI:   converted_mei/iiif_tutorial/bsb00023199_00185.mei
Image:        converted_mei/iiif_tutorial/img/bsb00023199_00185.jpg
Annotation:   converted_mei/iiif_tutorial/bsb00023199_00185_measure_annotations.xml
Final output: converted_mei/iiif_tutorial/bsb00023199_00185_facs_zones.mei
Detector:     https://measure-detector.edirom.de/upload
Run enabled:  True


## 3. Run the integration

With the flag enabled this cell:

1. copies the source to the working name in `TARGET_DIR` (original file unchanged);
2. downloads the facsimile JPEG (`IIIF_IMAGE_URL` if set, otherwise the BSB IIIF URL for the working stem);
3. uploads that picture to the measure-detector network and integrates the returned zones, first with a local `img/...` graphic target;
4. checks that the local image bytes match the IIIF URL that was fetched;
5. rewrites `<graphic @target>` to that IIIF URL in `{stem}_facs_zones.mei`.

If an annotation XML already exists and you do not want another detector upload, set `REUSE_ANNOTATIONS = True`.

In [6]:
if not RUN_IIIF_INTEGRATION:
    print("Skipped. Review the plan, then set RUN_IIIF_INTEGRATION = True.")
else:
    staged_mei = stage_mei_copy(
        source_mei,
        target_dir,
        target_stem,
        overwrite=OVERWRITE_STAGED_MEI,
    )
    print(f"Staged {display_path(source_mei)} -> {display_path(staged_mei)}")

    image_path, expected_iiif_url = download_facsimile_image(
        image_path,
        stem=target_stem,
        image_url=iiif_image_url,
        target_dpi=TARGET_DPI,
        page_width_mm=PAGE_WIDTH_MM,
        timeout=TIMEOUT,
        overwrite=False,
    )
    width, height = read_image_size(image_path)
    print(f"Image: {display_path(image_path)} ({width} x {height})")
    print(expected_iiif_url)

    annotation_path, local_output = detect_and_integrate_mei(
        staged_mei,
        image_dir=image_dir,
        detector_url=DETECTOR_URL,
        timeout=TIMEOUT,
        retries=2,
        retry_delay=2.0,
        minimum_measures=MINIMUM_MEASURES,
        max_measure_mismatch=MAX_MEASURE_MISMATCH,
        annotation_suffix="_measure_annotations.xml",
        output_suffix="_facs_zones",
        reuse_annotations=REUSE_ANNOTATIONS,
        overwrite=OVERWRITE_OUTPUT,
        graphic_target_mode="local",
        iiif_url_template=IIIF_IMAGE_URL_TEMPLATE,
        graphic_target=expected_iiif_url,
    )
    print(f"Annotation: {display_path(annotation_path)}")
    print(f"Local output: {display_path(local_output)}")

    local_hash = sha256_file(image_path)
    response = requests.get(expected_iiif_url, timeout=TIMEOUT)
    response.raise_for_status()
    if local_hash != sha256_bytes(response.content):
        raise RuntimeError("Local facsimile does not match the IIIF source")
    print("Local facsimile matches IIIF source")

    annotation_path, final_mei = detect_and_integrate_mei(
        staged_mei,
        image_dir=image_dir,
        detector_url=DETECTOR_URL,
        timeout=TIMEOUT,
        retries=2,
        retry_delay=2.0,
        minimum_measures=MINIMUM_MEASURES,
        max_measure_mismatch=MAX_MEASURE_MISMATCH,
        annotation_suffix="_measure_annotations.xml",
        output_suffix="_facs_zones",
        reuse_annotations=True,
        overwrite=True,
        graphic_target_mode="iiif",
        iiif_url_template=IIIF_IMAGE_URL_TEMPLATE,
        graphic_target=expected_iiif_url,
    )
    target, mei_width, mei_height = parse_graphic_from_output_mei(final_mei)
    final_model = read_facsimile_model(final_mei)
    assert target == expected_iiif_url, (target, expected_iiif_url)
    assert (mei_width, mei_height) == (width, height)
    assert final_model["has_facsimile"]
    assert final_model["zones"]
    assert final_model["linked"]
    print("Final MEI is ready")
    print(format_facsimile_summary(final_model))
    print(f"Target: {target}")

Staged test_corpus/Buxtehude-Anhang-S._185_musicxml_verovio.mei -> converted_mei/iiif_tutorial/bsb00023199_00185.mei
Image: converted_mei/iiif_tutorial/img/bsb00023199_00185.jpg (4134 x 5495)
https://api.digitale-sammlungen.de/iiif/image/v2/bsb00023199_00185/full/4134,/0/default.jpg
  uploading image: bsb00023199_00185.jpg
  wrote annotations: bsb00023199_00185_measure_annotations.xml
  wrote integrated MEI: bsb00023199_00185_facs_zones.mei
Annotation: converted_mei/iiif_tutorial/bsb00023199_00185_measure_annotations.xml
Local output: converted_mei/iiif_tutorial/bsb00023199_00185_facs_zones.mei
Local facsimile matches IIIF source
  reusing annotations: bsb00023199_00185_measure_annotations.xml
  wrote integrated MEI: bsb00023199_00185_facs_zones.mei
Final MEI is ready
MEI:             converted_mei/iiif_tutorial/bsb00023199_00185_facs_zones.mei
Viewer mode:     score + facsimile
Surfaces:        1
Measures:        17
Linked zones:    17
Missing @facs:   0
Target: https://api.digitale-s

## Notes for the next file

1. Change `SOURCE_MEI`. Pair it with `ARCHIVE_URL` and/or paste `IIIF_IMAGE_URL`.
2. Review the plan, then set `RUN_IIIF_INTEGRATION = True`.
3. If the staged working copy already exists and should be replaced, set `OVERWRITE_STAGED_MEI = True`.
4. To rerun measure detection, set `REUSE_ANNOTATIONS = False` and `OVERWRITE_OUTPUT = True`.
5. Inspect the result in [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb), then continue with consistency and schema checks.
6. For several files, use [`mei_batch_iiif_integration.ipynb`](mei_batch_iiif_integration.ipynb).